# AI for Detecting Concept Drift in Human Beliefs

This notebook implements an end‑to‑end framework for detecting ideological and semantic belief drift in text streams using NLP, clustering, drift detection, Bayesian modeling, and simulation.

## Setup & Libraries

In [ ]:
!pip install pandas numpy scikit-learn sentence-transformers hdbscan umap-learn river bertopic pymc plotly networkx langdetect

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## 1. Data Ingestion

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

texts_early = [
    "AI will help humanity",
    "AI will improve medicine",
    "AI will create new jobs",
    "AI technology is exciting",
    "AI will boost productivity"
]

texts_mid = [
    "AI might replace workers",
    "AI automation threatens jobs",
    "AI companies are too powerful",
    "AI may cause unemployment",
    "AI regulation is necessary"
]

texts_late = [
    "AI could be dangerous",
    "AI might threaten humanity",
    "AI safety is critical",
    "AI needs strict regulation",
    "AI could become uncontrollable"
]

rows = []

# optimistic phase
for i in range(200):
    rows.append({
        "text": np.random.choice(texts_early),
        "timestamp": pd.Timestamp("2023-01-01") + pd.Timedelta(days=np.random.randint(0,60)),
        "subreddit": "technology"
    })

# economic concern phase
for i in range(200):
    rows.append({
        "text": np.random.choice(texts_mid),
        "timestamp": pd.Timestamp("2023-03-01") + pd.Timedelta(days=np.random.randint(0,60)),
        "subreddit": "technology"
    })

# risk phase
for i in range(200):
    rows.append({
        "text": np.random.choice(texts_late),
        "timestamp": pd.Timestamp("2023-05-01") + pd.Timedelta(days=np.random.randint(0,60)),
        "subreddit": "futurology"
    })

data = pd.DataFrame(rows)

data.head()

In [ ]:
# Basic cleaning
import re
from langdetect import detect

def clean_text(t):
    t = re.sub(r"http\S+", "", str(t))
    t = re.sub(r"[^A-Za-z0-9 .,!?]", "", t)
    return t

data['text'] = data['text'].apply(clean_text)

def is_english(t):
    try:
        return detect(t) == 'en'
    except:
        return False

data = data[data['text'].apply(is_english)]

## 2. Belief Representation (Semantic Embeddings)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-mpnet-base-v2')

texts = data['text'].tolist()
embeddings = model.encode(texts, show_progress_bar=True)

embeddings[:2]

## 3. Belief Clustering

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size=50)
clusters = clusterer.fit_predict(embeddings)

data['cluster'] = clusters

data['cluster'].value_counts()

## 4. Temporal Modeling of Beliefs

In [ ]:
data['month'] = data['timestamp'].dt.to_period('M')

belief_trends = data.groupby(['month','cluster']).size().unstack(fill_value=0)

belief_trends.head()

In [ ]:
belief_trends.plot(figsize=(12,6))
plt.title("Belief Cluster Frequency Over Time")
plt.show()

## 5. Concept Drift Detection

In [ ]:
from scipy.spatial.distance import jensenshannon

drifts = []

for i in range(len(belief_trends)-1):

    p = belief_trends.iloc[i] / belief_trends.iloc[i].sum()
    q = belief_trends.iloc[i+1] / belief_trends.iloc[i+1].sum()

    dist = jensenshannon(p,q)

    if dist > 0.2:
        drifts.append(i)

drift_months = belief_trends.index[drifts]
drift_months

## 6. Ideological Vector Field

In [ ]:
from sklearn.metrics.pairwise import cosine_distances

centroids = []

for m in data['month'].unique():
    subset = embeddings[data['month'] == m]
    centroids.append(subset.mean(axis=0))

drift_velocity = []

for i in range(len(centroids)-1):
    drift_velocity.append(
        cosine_distances([centroids[i]],[centroids[i+1]])[0][0]
    )

drift_velocity

## 7. Causal Trigger Detection via Topic Modeling

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic()

topics, probs = topic_model.fit_transform(texts)

topic_model.get_topic_info().head()

## 8. Bayesian Belief Updating

In [ ]:
import pymc as pm

observed = np.array(stream)

with pm.Model() as model:

    belief_prior = pm.Normal("belief_prior",0,1)

    sigma = pm.HalfNormal("sigma",1)

    likelihood = pm.Normal("obs",belief_prior,sigma,observed=observed)

    trace = pm.sample(500, tune=500, chains=2)


## 9. Visualization (UMAP Semantic Map)

In [ ]:
import umap

reducer = umap.UMAP()

embedding_2d = reducer.fit_transform(embeddings)

plt.figure(figsize=(8,6))
plt.scatter(embedding_2d[:,0],embedding_2d[:,1],c=clusters,s=5)
plt.title("Belief Semantic Landscape")
plt.show()

## 10. Evaluation Metrics

In [ ]:
from scipy.spatial.distance import jensenshannon

distances = []

for i in range(len(belief_trends)-1):
    p = belief_trends.iloc[i] / belief_trends.iloc[i].sum()
    q = belief_trends.iloc[i+1] / belief_trends.iloc[i+1].sum()

    distances.append(jensenshannon(p,q))

distances

## 11. Simulation of Belief Systems

In [ ]:
n_agents = 500
beliefs = np.random.normal(0,1,n_agents)

history = []

for t in range(50):

    social = np.mean(beliefs)
    noise = np.random.normal(0,0.1,n_agents)

    beliefs = np.tanh(beliefs + 0.1*social + noise)

    history.append(beliefs.mean())

plt.plot(history)
plt.title("Simulated Belief Evolution")
plt.show()

In [ ]:
# how many clusters (excluding noise)
clusters = sorted([c for c in data['cluster'].unique() if c != -1])
print(f"Discovered {len(clusters)} belief clusters.")

# drift events
if len(drift_months) > 0:
    print(f"Concept drift detected at {len(drift_months)} time point(s):")
    print(list(drift_months.astype(str)))
else:
    print("No concept drift detected.")

# semantic drift strength
print(f"Average semantic shift between periods: {np.mean(drift_velocity):.3f}")